# Lezione 05 — Classificazione di testi

**Corso:** NLP Classico con Python · **Blocco 2:** Rappresentare e classificare
**Tempo stimato:** 110 minuti · **Capitolo del manuale:** `manual/05_clasificacion_de_textos_IT.md`

## Cosa vedrai in questa lezione

- **Naive Bayes multinomiale** calcolato **a mano** su un esempio minuscolo, con il lisciamento di Laplace, e verificato con `MultinomialNB`.
- **Regressione logistica** e **SVM lineare** (`LinearSVC`) su TF-IDF, confrontate con Naive Bayes su ticket in spagnolo e in italiano.
- La **`Pipeline`** e la **fuga di informazione**: un esperimento reale, con il vettorizzatore fuori e dentro la `Pipeline`.
- **Metriche:** precisione, richiamo (*recall*), F1 (macro e ponderato) e matrice di confusione, calcolate a mano e con `classification_report`.
- **Classi sbilanciate:** creiamo lo sbilanciamento sottocampionando e vediamo perché l'accuratezza inganna.
- **Leggere i coefficienti** del modello, con senso critico: i ticket hanno artefatti che finiscono tra le parole più pesanti.
- **Validazione incrociata stratificata** (`StratifiedKFold`).

## Prerequisiti

- Sacco di parole e TF-IDF con `TfidfVectorizer` (lezione 4) e perché `fit` si fa solo con l'addestramento.
- Python con `numpy` e `pandas`. Librerie: `scikit-learn`, `matplotlib`. La cella seguente installa solo ciò che manca (funziona allo stesso modo in Google Colab).
- La cartella `datos/` del corso (`tickets_es.csv` e `tickets_it.csv`). In Colab, caricala dal pannello dei file.

> **Come usare questo notebook:** esegui le celle in ordine. I calcoli a mano si verificano con scikit-learn tramite un `assert`. **Avviso:** i ticket in spagnolo e in italiano sono adattamenti molto stretti l'uno dell'altro, quindi **non sono due esperimenti indipendenti**; e sono corpora piccoli e sintetici scritti per il corso (*esempio illustrativo*): le cifre non si generalizzano a testi reali.

---
## Installazione

Installa solo ciò che manca. In un ambiente con tutto installato non fa nulla.

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("scipy", "scipy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("scikit-learn", "sklearn")
print("Librerie pronte.")

---
## Cella 0 — Setup

Importiamo le librerie, fissiamo il **seme**, definiamo `cargar_datos` (cerca `datos/`, `../datos/` e `/content/datos/`) e leggiamo i 400 ticket di ciascuna lingua (100 per categoria: `cuenta`, `devoluciones`, `envio`, `facturacion`). Dividiamo **una sola volta** in 75 % addestramento e 25 % test, **stratificato** per categoria, in modo che il test abbia 25 ticket di ciascuna. Definiamo anche `crear_pipeline`, che useremo per tutta la lezione.

In [ ]:
import os
import re
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

random.seed(42)                          # seme fisso: tutti otteniamo gli stessi risultati
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = False
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

ARCHIVO_TICKETS = {"spa": "tickets_es.csv", "ita": "tickets_it.csv"}
tickets = {idioma: pd.read_csv(cargar_datos(archivo)) for idioma, archivo in ARCHIVO_TICKETS.items()}
ETIQUETAS = sorted(tickets["spa"]["categoria"].unique())      # ordine alfabetico, lo stesso che usa scikit-learn


def dividir(df):
    """75 % addestramento / 25 % test, stratificato per categoria."""
    return train_test_split(df["texto"], df["categoria"], test_size=0.25, stratify=df["categoria"], random_state=42)


DIV = {idioma: dividir(df) for idioma, df in tickets.items()}


def crear_pipeline(clasificador, **opciones):
    """TF-IDF (unigrammi e bigrammi) + classificatore, in un'unica Pipeline."""
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), **opciones)), ("clf", clasificador)])


def separar_veredicto(texto):
    """Separa l'ultima frase se ha 4 parole o meno (verdetto o etichetta di oggetto)."""
    m = re.search(r"(?<=[.?!])\s+([^.?!]+[.?!])\s*$", texto)
    if m and len(m.group(1).split()) <= 4:
        return texto[:m.start()].strip(), m.group(1).strip()
    return texto, ""


print({idioma: (len(d[0]), len(d[1])) for idioma, d in DIV.items()}, "(addestramento, test)")
print("Dati e librerie caricati correttamente.")

---
## 1. Punto di partenza: cos'è un buon risultato

Un classificatore supervisionato impara da esempi con etichetta (`texto` → `categoria`) e si valuta su esempi che **non ha mai visto**. Prima di addestrare qualsiasi cosa conviene fissare una **linea di base**: il risultato di un modello banale. Se un modello non la supera con chiarezza, non ha imparato nulla. Qui, un `DummyClassifier` che predice sempre la stessa classe.

In [ ]:
for idioma in ("spa", "ita"):
    X_ent, X_prueba, y_ent, y_prueba = DIV[idioma]
    trivial = DummyClassifier(strategy="most_frequent").fit(X_ent, y_ent)
    print(idioma, "addestramento:", dict(y_ent.value_counts().sort_index()),
          "| test:", dict(y_prueba.value_counts().sort_index()))
    print("   accuratezza del modello banale:", round(trivial.score(X_prueba, y_prueba), 3))
    assert y_prueba.value_counts().nunique() == 1          # 25 ticket per categoria nel test

**Cosa osservi?** Con quattro categorie bilanciate, indovinare sempre la stessa dà 0,25: questa è la linea di base. Tutto ciò che faremo deve superare quel 25 % con ampio margine. Nel test ci sono 100 ticket: **ogni ticket vale un punto percentuale**, quindi differenze di uno o due punti tra modelli sono rumore.

---
## 2. Naive Bayes multinomiale a mano, con lisciamento di Laplace

Naive Bayes sceglie la classe con il massimo `P(classe) · Π P(parola | classe)`, supponendo che le parole siano indipendenti all'interno di ogni classe («ingenuo»). Ogni `P(parola | classe)` si ottiene contando le parole nei testi di quella classe. Problema: una parola **mai vista** in una classe ha probabilità 0 e annulla l'intero prodotto. Il **lisciamento di Laplace** somma `alpha = 1` a ogni conteggio:

`P(parola | classe) = (conteggio + alpha) / (N_classe + alpha · V)`

con `N_classe` il totale di parole della classe e `V` la dimensione del vocabolario. Lo calcoliamo con un corpus **minuscolo** (due ticket per classe) e lo verifichiamo con `MultinomialNB(alpha=1)`.

In [ ]:
# datos-es
MINI = {
    "spa": {"facturacion": ["cobro doble tarjeta", "factura error cobro"],
            "envio": ["paquete perdido no llega", "no llega el envío"]},
    "ita": {"facturacion": ["addebito doppio carta", "fattura errore addebito"],
            "envio": ["pacco perso non arriva", "non arriva la spedizione"]},
}
PRUEBA = {"spa": "cobro no llega", "ita": "addebito non arriva"}

In [ ]:
def nb_a_mano(mini, texto, alpha=1.0):
    """Naive Bayes multinomiale con lisciamento: restituisce priori, tabella di probabilità e posteriori."""
    docs = [(clase, re.findall(r"\w+", d.lower())) for clase, ds in mini.items() for d in ds]
    vocab = sorted({t for _, toks in docs for t in toks})
    V = len(vocab)
    palabras = re.findall(r"\w+", texto.lower())
    clases = sorted(mini)
    resultado = {}
    for clase in clases:
        toks = [t for c, ts in docs if c == clase for t in ts]
        N = len(toks)
        cuenta = Counter(toks)
        priori = sum(1 for c, _ in docs if c == clase) / len(docs)
        probs = {p: (cuenta[p] + alpha) / (N + alpha * V) for p in palabras}
        verosimilitud = priori * np.prod(list(probs.values()))
        resultado[clase] = {"N": N, "V": V, "priori": priori, "probs": probs, "recuentos": {p: cuenta[p] for p in palabras}, "puntuacion": verosimilitud}
    total = sum(r["puntuacion"] for r in resultado.values())
    for r in resultado.values():
        r["posterior"] = r["puntuacion"] / total if total > 0 else float("nan")      # 0/0 senza lisciamento
    return resultado


for idioma in ("spa", "ita"):
    res = nb_a_mano(MINI[idioma], PRUEBA[idioma])
    print("Ticket nuovo:", repr(PRUEBA[idioma]))
    for clase, r in res.items():
        detalle = " · ".join(f"P({p}|{clase[:3]}) = {r['recuentos'][p] + 1}/{r['N'] + r['V']}" for p in r["probs"])
        print(f"  {clase:12s} N = {r['N']}, V = {r['V']}, priori = {r['priori']:.2f} | {detalle}")
        print(f"               punteggio = {r['puntuacion']:.6f}  →  posteriore = {r['posterior']:.4f}")

    # Senza lisciamento (alpha = 0): i prodotti si annullano
    sin = nb_a_mano(MINI[idioma], PRUEBA[idioma], alpha=0.0)
    print("  Senza lisciamento (alpha = 0), punteggi:", {k: v["puntuacion"] for k, v in sin.items()})

    # Verifica con scikit-learn
    textos = [d for ds in MINI[idioma].values() for d in ds]
    etiquetas = [c for c, ds in MINI[idioma].items() for _ in ds]
    cv = CountVectorizer(token_pattern=r"\w+")
    nb = MultinomialNB(alpha=1.0).fit(cv.fit_transform(textos), etiquetas)
    proba = nb.predict_proba(cv.transform([PRUEBA[idioma]]))[0]
    for clase, p in zip(nb.classes_, proba):
        assert np.isclose(p, res[clase]["posterior"])
    assert res["envio"]["N"] == 8 and res["facturacion"]["N"] == 6 and res["envio"]["V"] == 11
    print("  coincide con MultinomialNB(alpha=1) ✔, predizione:", nb.predict(cv.transform([PRUEBA[idioma]]))[0])

**Cosa osservi?** In italiano, `facturacion` ha 6 parole e `envio` 8; il vocabolario ne ha 11. `addebito` compare 2 volte in `facturacion`: `(2 + 1)/(6 + 11) = 3/17`. Le altre due parole del ticket nuovo non compaiono in quella classe e valgono `1/17` ciascuna (**il lisciamento evita lo zero**). I posteriori sono `facturacion` 0,3176 e `envio` 0,6824: vince `envio`, perché `non arriva` pesa più di `addebito`. Con `alpha = 0` **entrambi i punteggi sono 0** e non si può decidere. Un dettaglio: con testi lunghi i prodotti diventano minuscoli; in pratica si sommano i **logaritmi** (scikit-learn lo fa così internamente).

---
## 3. Tre classificatori su TF-IDF

I tre condividono la stessa rappresentazione (`TfidfVectorizer` con unigrammi e bigrammi) e cambiano solo il classificatore:

- **`MultinomialNB`**: probabilistico, molto veloce; definito sui conteggi, ma con il TF-IDF funziona bene nella pratica.
- **`LogisticRegression`**: impara un peso per parola e classe; dà probabilità e coefficienti che si possono leggere.
- **`LinearSVC`** (SVM lineare): cerca l'iperpiano che separa le classi con il margine maggiore; di solito è molto competitivo con il testo.

Li mettiamo in una `Pipeline` perché il vettorizzatore e il classificatore si adattino insieme e solo con l'addestramento.

In [ ]:
MODELOS = {"MultinomialNB": lambda: MultinomialNB(),
           "LogisticRegression": lambda: LogisticRegression(max_iter=1000),
           "LinearSVC": lambda: LinearSVC()}
PREDICCIONES = {}                                  # salviamo le predizioni per usarle dopo
filas = []
for nombre, fabrica in MODELOS.items():
    fila = {"modello": nombre}
    for idioma in ("spa", "ita"):
        X_ent, X_prueba, y_ent, y_prueba = DIV[idioma]
        modelo = crear_pipeline(fabrica()).fit(X_ent, y_ent)
        pred = modelo.predict(X_prueba)
        PREDICCIONES[(nombre, idioma)] = pred
        fila[f"accuratezza {idioma}"] = round(accuracy_score(y_prueba, pred), 3)
        fila[f"F1 macro {idioma}"] = round(f1_score(y_prueba, pred, average="macro"), 3)
        assert accuracy_score(y_prueba, pred) > 0.6         # ben sopra il 25 % del modello banale
    filas.append(fila)
tabla_modelos = pd.DataFrame(filas)
print(tabla_modelos.to_string(index=False))

**Cosa osservi?** I tre modelli stanno tra 0,85 e 0,91 di accuratezza, molto sopra 0,25. La differenza massima tra modelli, nella stessa lingua, è di 0,04: **pochi ticket su 100**. Con una sola partizione di test non si può affermare che uno sia «migliore»; più avanti lo guardiamo con la validazione incrociata. E ricorda che spagnolo e italiano sono adattamenti stretti: che le cifre si somiglino **non** conferma nulla da solo.

---
## 4. `Pipeline` e fuga di informazione: un esperimento reale

**Fuga di informazione** (*data leakage*): informazione del test che si infiltra nell'addestramento e rende la valutazione troppo ottimistica. L'errore classico è adattare il vettorizzatore su **tutto** il corpus e **poi** dividere. La `Pipeline` lo evita: a ogni `fit` adatta il vettorizzatore solo con i dati che riceve (e dentro la validazione incrociata, solo con l'addestramento di ogni partizione).

Misuriamolo. Confrontiamo, con **20 partizioni casuali** diverse (semi da 0 a 19), l'accuratezza con il TF-IDF **adattato su tutto il corpus** (`ajuste global`) rispetto alla `Pipeline` corretta, con 40 e con 300 ticket di addestramento. È un esperimento più lungo: richiede qualche secondo.

In [ ]:
def fuga_vectorizador(df, n_ent, semillas=range(20)):
    """Accuratezza media con vettorizzatore adattato su tutto il corpus contro Pipeline corretta."""
    global_, correcta = [], []
    for semilla in semillas:
        X_ent, X_prueba, y_ent, y_prueba = train_test_split(df["texto"], df["categoria"], train_size=n_ent, test_size=100,
                                                             stratify=df["categoria"], random_state=semilla)
        vec = TfidfVectorizer(ngram_range=(1, 2)).fit(df["texto"])          # ERRATO: vede anche il test
        modelo = LogisticRegression(max_iter=1000).fit(vec.transform(X_ent), y_ent)
        global_.append(modelo.score(vec.transform(X_prueba), y_prueba))
        correcta.append(crear_pipeline(LogisticRegression(max_iter=1000)).fit(X_ent, y_ent).score(X_prueba, y_prueba))
    return np.mean(global_), np.mean(correcta)


filas = []
for idioma in ("spa", "ita"):
    for n_ent in (40, 300):
        g, p = fuga_vectorizador(tickets[idioma], n_ent)
        filas.append({"lingua": idioma, "addestramento": n_ent, "ajuste global": round(g, 4), "Pipeline": round(p, 4), "differenza": round(g - p, 4)})
        assert abs(g - p) < 0.03                       # con il TF-IDF la differenza è piccola (lo verifichiamo, non lo diamo per scontato)
print(pd.DataFrame(filas).to_string(index=False))

**Cosa osservi?** Con il TF-IDF, **la differenza è praticamente nulla** (qualche millesimo, a volte negativa): il vettorizzatore impara solo il vocabolario e gli `idf`, **senza guardare le etichette**, e con questi corpora il vocabolario globale cambia a malapena ciò che il modello può imparare. È un risultato onesto: non sempre la fuga si vede. Ma questo **non** giustifica la scorciatoia: il rischio reale compare appena un passo della catena **usa le etichette**. Lo vediamo nella cella seguente.

### Un passo che usa davvero le etichette: selezione delle caratteristiche

Aggiungiamo `SelectKBest(chi2, k=50)`, che tiene le 50 parole più associate alle etichette. Perché l'effetto sia inequivocabile **mescoliamo le etichette a caso**: non c'è più alcuna relazione reale tra testo e categoria, e con quattro classi un modello onesto non dovrebbe superare il 25 % di risposte corrette. Confrontiamo il selezionare su **tutto** il corpus e **poi** dividere, rispetto al farlo dentro la `Pipeline`.

In [ ]:
def fuga_seleccion(df, semillas=range(20)):
    """Con etichette mescolate: selezione delle caratteristiche fuori e dentro la Pipeline."""
    rng = np.random.RandomState(0)
    y_falsa = pd.Series(rng.permutation(df["categoria"].values), index=df.index)
    X_todo = TfidfVectorizer(ngram_range=(1, 2)).fit_transform(df["texto"])
    X_sel = SelectKBest(chi2, k=50).fit_transform(X_todo, y_falsa)              # ERRATO: sceglie le parole guardando TUTTE le etichette
    fuera, dentro = [], []
    for semilla in semillas:
        idx_ent, idx_prueba = train_test_split(np.arange(len(df)), test_size=0.25, stratify=y_falsa, random_state=semilla)
        modelo = LogisticRegression(max_iter=1000).fit(X_sel[idx_ent], y_falsa.iloc[idx_ent])
        fuera.append(modelo.score(X_sel[idx_prueba], y_falsa.iloc[idx_prueba]))
        pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))), ("sel", SelectKBest(chi2, k=50)), ("clf", LogisticRegression(max_iter=1000))])
        dentro.append(pipe.fit(df["texto"].iloc[idx_ent], y_falsa.iloc[idx_ent]).score(df["texto"].iloc[idx_prueba], y_falsa.iloc[idx_prueba]))
    return np.mean(fuera), np.mean(dentro)


filas = []
for idioma in ("spa", "ita"):
    fuera, dentro = fuga_seleccion(tickets[idioma])
    filas.append({"lingua": idioma, "selezione fuori dalla Pipeline": round(fuera, 3),
                  "selezione dentro la Pipeline": round(dentro, 3), "caso": 0.25})
    assert fuera > dentro + 0.05                       # la fuga gonfia il risultato in modo netto
print(pd.DataFrame(filas).to_string(index=False))

**Cosa osservi?** Con etichette **casuali** (senza nulla da imparare), il modello con la selezione fatta fuori azzecca 0,411 in spagnolo e 0,443 in italiano: **molto più del caso (0,25)**, e solo perché le 50 parole sono state scelte guardando anche le etichette del test. Dentro la `Pipeline` il risultato torna intorno al caso (0,253 e 0,258), ed è quello corretto. **Regola pratica:** ogni passo che impara qualcosa dai dati (vettorizzatore, selezione delle caratteristiche, scalatura, riduzione di dimensione) va **dentro** la `Pipeline`.

---
## 5. Metriche: precisione, richiamo (*recall*), F1 e matrice di confusione

Per ogni classe, a partire dalla **matrice di confusione** (righe = classe reale, colonne = classe predetta):

- **Precisione** = risposte corrette della classe / tutto ciò che il modello ha etichettato come quella classe (somma della colonna).
- **Richiamo** (*recall*) = risposte corrette della classe / tutti quelli che sono realmente di quella classe (somma della riga).
- **F1** = media armonica di precisione e richiamo: `2 · P · R / (P + R)`.
- **F1 macro** = media semplice degli F1 per classe (tutte pesano uguale). **F1 ponderato** = media pesata per il numero di esempi reali di ogni classe (il *supporto*).

Prima, a mano, con una piccola matrice di 10 predizioni.

In [ ]:
CLASES_M = ["cuenta", "devoluciones", "envio"]
M = np.array([[4, 1, 0],          # reale cuenta: 4 giusti, 1 confuso con devoluciones
              [1, 2, 0],          # reale devoluciones
              [0, 1, 1]])         # reale envio
aciertos = np.diag(M)
precision = aciertos / M.sum(axis=0)
recall = aciertos / M.sum(axis=1)
f1 = 2 * precision * recall / (precision + recall)
soporte = M.sum(axis=1)
f1_macro_mano = f1.mean()
f1_pond_mano = (f1 * soporte).sum() / soporte.sum()
print(pd.DataFrame({"precisione": precision.round(4), "recall": recall.round(4), "F1": f1.round(4), "supporto": soporte}, index=CLASES_M))
print("F1 macro:", round(f1_macro_mano, 4), "| F1 ponderato:", round(f1_pond_mano, 4), "| accuratezza:", aciertos.sum() / M.sum())

# Verifica: ricostruiamo le etichette dalla matrice e usiamo scikit-learn
y_real = [CLASES_M[i] for i in range(3) for j in range(3) for _ in range(M[i, j])]
y_pred = [CLASES_M[j] for i in range(3) for j in range(3) for _ in range(M[i, j])]
p, r, f, s = precision_recall_fscore_support(y_real, y_pred, labels=CLASES_M)
assert np.allclose(p, precision) and np.allclose(r, recall) and np.allclose(f, f1) and (s == soporte).all()
assert np.isclose(f1_score(y_real, y_pred, average="macro"), f1_macro_mano)
assert np.isclose(f1_score(y_real, y_pred, average="weighted"), f1_pond_mano)
print("Coincide con scikit-learn ✔")

**Cosa osservi?** `envio` ha precisione 1,0 (ciò che ha segnato come `envio` era `envio`) ma richiamo 0,5 (ne ha trovato solo 1 su 2). Sono cose diverse: la precisione guarda **ciò che il modello dice**, il richiamo **ciò che esiste**. L'F1 macro (0,6794) e quello ponderato (0,7048) differiscono perché `cuenta` (F1 più alto) ha più peso nel ponderato. Ora li applichiamo ai ticket reali, con `classification_report` e la matrice di confusione del modello di regressione logistica.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4.5))
for eje, idioma in zip(ejes, ("spa", "ita")):
    y_prueba = DIV[idioma][3]
    pred = PREDICCIONES[("LogisticRegression", idioma)]
    print("=== ", idioma, " ===")
    print(classification_report(y_prueba, pred, digits=3, zero_division=0))
    matriz = confusion_matrix(y_prueba, pred, labels=ETIQUETAS)
    assert matriz.sum() == len(y_prueba) and (matriz.sum(axis=1) == 25).all()
    im = eje.imshow(matriz, cmap="Blues")
    eje.set_xticks(range(4)); eje.set_xticklabels(ETIQUETAS, rotation=30)
    eje.set_yticks(range(4)); eje.set_yticklabels(ETIQUETAS)
    for i in range(4):
        for j in range(4):
            eje.text(j, i, matriz[i, j], ha="center", va="center", color="white" if matriz[i, j] > 15 else "black")
    eje.set_xlabel("classe predetta"); eje.set_ylabel("classe reale")
    eje.set_title(f"Matrice di confusione ({idioma})")
    fuera_diag = matriz.copy(); np.fill_diagonal(fuera_diag, 0)
    i, j = np.unravel_index(fuera_diag.argmax(), fuera_diag.shape)
    print("Confusione più frequente:", ETIQUETAS[i], "→", ETIQUETAS[j], f"({fuera_diag[i, j]} ticket)")
plt.tight_layout()
plt.show()

**Cosa osservi?** Poiché il test è bilanciato (25 per classe), accuratezza, F1 macro e ponderato **coincidono** o quasi. La diagonale della matrice concentra la maggior parte dei ticket; gli errori si distribuiscono e mostrano quali categorie si confondono: in spagnolo, l'errore più frequente è `devoluciones` → `facturacion` (4 ticket, perché un rimborso suona come denaro e addebito); in italiano nessuna coppia supera i 2 ticket. Ricorda che il 10-15 % dei ticket contiene parole tipiche di un'altra categoria: parte di questi errori sta nei dati, non nel modello. **Quando le classi non sono bilanciate, questo cambia:** è il punto successivo.

---
## 6. Classi sbilanciate: l'accuratezza inganna

I ticket sono bilanciati, quindi **lo sbilanciamento lo creiamo noi**, sottocampionando ogni categoria a un numero diverso (100 di `facturacion`, 50 di `envio`, 20 di `cuenta` e 10 di `devoluciones`; non si crea nessun file nuovo). È un esperimento guidato: addestriamo con e senza `class_weight="balanced"` e guardiamo accuratezza, F1 macro e ponderato e il richiamo di ogni classe.

In [ ]:
CUOTAS = {"facturacion": 100, "envio": 50, "cuenta": 20, "devoluciones": 10}


def submuestrear(df, cuotas, semilla=42):
    """Tiene cuotas[categoria] ticket per ogni categoria."""
    partes = [g.sample(n=cuotas[c], random_state=semilla) for c, g in df.groupby("categoria")]
    return pd.concat(partes).reset_index(drop=True)


filas, informes = [], {}
for idioma in ("spa", "ita"):
    des = submuestrear(tickets[idioma], CUOTAS)
    assert des["categoria"].value_counts().to_dict() == CUOTAS
    X_ent, X_prueba, y_ent, y_prueba = train_test_split(des["texto"], des["categoria"], test_size=0.3, stratify=des["categoria"], random_state=42)
    trivial = DummyClassifier(strategy="most_frequent").fit(X_ent, y_ent).predict(X_prueba)
    for peso in (None, "balanced"):
        modelo = crear_pipeline(LogisticRegression(max_iter=1000, class_weight=peso)).fit(X_ent, y_ent)
        pred = modelo.predict(X_prueba)
        informes[(idioma, peso)] = classification_report(y_prueba, pred, digits=3, zero_division=0)
        _, recall_clase, _, _ = precision_recall_fscore_support(y_prueba, pred, labels=ETIQUETAS, zero_division=0)
        fila = {"lingua": idioma, "class_weight": str(peso), "accuratezza": round(accuracy_score(y_prueba, pred), 3),
                "F1 macro": round(f1_score(y_prueba, pred, average="macro"), 3), "F1 ponderato": round(f1_score(y_prueba, pred, average="weighted"), 3)}
        fila.update({f"recall {c}": round(r, 2) for c, r in zip(ETIQUETAS, recall_clase)})
        filas.append(fila)
    print(idioma, "test:", dict(y_prueba.value_counts().sort_index()), "| accuratezza sempre «facturacion»:", round(accuracy_score(y_prueba, trivial), 3))
print(pd.DataFrame(filas).to_string(index=False))
print()
print("Rapporto completo, spagnolo, senza pesi:")
print(informes[("spa", None)])

**Cosa osservi?** Senza pesi, l'**accuratezza sembra accettabile** (0,704 in spagnolo, 0,667 in italiano; dire sempre «facturacion» darebbe già 0,556) ma l'**F1 macro crolla** (0,371 e 0,335) e il richiamo di `cuenta` e `devoluciones` è **0**: il modello non rileva mai le classi piccole. L'accuratezza le nasconde perché pesano poco. Con `class_weight="balanced"` l'F1 macro sale a 0,796 e 0,776. **Attenzione:** il test ha solo 6 ticket di `cuenta` e 3 di `devoluciones`, quindi ciascuno sposta molto le cifre; lo confermiamo con la validazione incrociata nella sezione 8. **Regola pratica:** con classi sbilanciate guarda l'F1 macro e il richiamo per classe, non solo l'accuratezza.

---
## 7. Leggere i coefficienti… con senso critico

Nella regressione logistica ogni parola ha un **coefficiente per classe**: i positivi spingono verso quella classe e i negativi contro. È uno dei suoi grandi vantaggi: si può leggere. Ma **leggere non è credere**. Guarda le parole più pesanti di ogni categoria, addestrando su tutto il corpus solo per ispezionare.

In [ ]:
def top_coeficientes(df, k=10):
    """Le k parole con il coefficiente maggiore di ogni classe (unigrammi)."""
    pipe = Pipeline([("tfidf", TfidfVectorizer()), ("clf", LogisticRegression(max_iter=1000))]).fit(df["texto"], df["categoria"])
    palabras = pipe.named_steps["tfidf"].get_feature_names_out()
    coef = pipe.named_steps["clf"].coef_
    return {clase: [(palabras[j], round(float(coef[i, j]), 2)) for j in np.argsort(coef[i])[::-1][:k]]
            for i, clase in enumerate(pipe.named_steps["clf"].classes_)}


TOP = {idioma: top_coeficientes(df) for idioma, df in tickets.items()}
for idioma in ("spa", "ita"):
    print("===", idioma, "===")
    for clase, lista in TOP[idioma].items():
        print(f"{clase:12s}", ", ".join(f"{p} ({v})" for p, v in lista))

**Cosa osservi?** La maggior parte sono parole sensate (`fattura`, `pacco`, `reso`, `password`…). Ma guarda quelle che **non** sono tematiche: parole funzionali come `mi`, `quiero`, `sin`, `non`, `voglio`, `in`, `la`. Riflettono i **modelli di scrittura** di ogni categoria (chi restituisce dice «voglio»), non il suo tema. E c'è dell'altro: i ticket hanno **artefatti**. Tra il 43 % e il 47 % termina con una frase molto corta, un «verdetto» (`Sin seguimiento.`, `Accesso bloccato.`), e circa il 6 % con un'etichetta di oggetto (`Asunto: cobro.`, `Oggetto: addebito.`). Verifichiamolo.

### Esperimento guidato: quanto si appoggia il modello sul verdetto?

Con `separar_veredicto` separiamo l'ultima frase se ha 4 parole o meno. Prima contiamo quanti ticket la hanno e quale parte delle parole più pesanti compare in quel verdetto; poi **togliamo il verdetto** e misuriamo di nuovo con la validazione incrociata.

In [ ]:
for idioma in ("spa", "ita"):
    df = tickets[idioma]
    partes = df["texto"].map(separar_veredicto)
    cuerpos = partes.map(lambda t: t[0])
    veredictos = partes.map(lambda t: t[1])
    con_veredicto = (veredictos != "").sum()
    con_asunto = df["texto"].str.contains(r"(?:Asunto|Oggetto):").sum()
    print(f"{idioma}: ticket con frase finale corta = {con_veredicto} ({100 * con_veredicto / len(df):.1f} %), con etichetta di oggetto = {con_asunto} ({100 * con_asunto / len(df):.1f} %)")
    assert con_veredicto > 100 and con_asunto > 10

    # Per le 8 parole più pesanti di ogni classe: in quanti ticket compaiono e in quanti sono nel verdetto?
    filas = []
    for clase, lista in TOP[idioma].items():
        for palabra, coef in lista[:8]:
            patron = r"\b" + re.escape(palabra) + r"\b"
            en_texto = df["texto"].str.lower().str.contains(patron)
            en_veredicto = veredictos.str.lower().str.contains(patron)
            filas.append({"classe": clase, "parola": palabra, "coef.": coef, "ticket con la parola": int(en_texto.sum()),
                          "nel verdetto": int(en_veredicto.sum())})
    tabla = pd.DataFrame(filas)
    tabla["% nel verdetto"] = (100 * tabla.iloc[:, 4] / tabla.iloc[:, 3]).round(0)
    print(tabla.sort_values("% nel verdetto", ascending=False).head(8).to_string(index=False))

    # Ablazione: validazione incrociata con e senza verdetto
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    acc_con = cross_val_score(crear_pipeline(LogisticRegression(max_iter=1000)), df["texto"], df["categoria"], cv=skf).mean()
    acc_sin = cross_val_score(crear_pipeline(LogisticRegression(max_iter=1000)), cuerpos, df["categoria"], cv=skf).mean()
    print(f"   accuratezza (CV a 5 partizioni) con il verdetto: {acc_con:.4f} | senza il verdetto: {acc_sin:.4f} | differenza: {acc_con - acc_sin:.4f}")
    assert acc_con > acc_sin

**Cosa osservi?** Diverse tra le parole più pesanti compaiono soprattutto in quelle frasi finali, e **togliendo il verdetto l'accuratezza scende** (0,8825 → 0,8475 in spagnolo; 0,9050 → 0,8550 in italiano). Il modello si appoggia su una scorciatoia che un ticket reale non offre quasi mai: nessuno chiude il proprio messaggio con `Tema de garantía.`. **Lezione:** un coefficiente alto dice cosa usa il modello, **non** cosa conta nel mondo reale. Prima di interpretare un elenco di parole pesanti, cerca gli artefatti (etichette, firme, modelli, date) e confronta il modello con e senza di essi. E ricorda: poiché i ticket in spagnolo e in italiano sono adattamenti stretti, questo risultato è **una sola osservazione vista due volte**, non una replica indipendente.

---
## 8. Validazione incrociata stratificata

Una sola partizione addestramento/test dà una cifra che dipende dalla fortuna del taglio. La **validazione incrociata** a `k` partizioni divide i dati in `k` parti e usa ciascuna come test una volta; quella **stratificata** (`StratifiedKFold`) mantiene in ogni parte la proporzione delle classi. Passare l'intera `Pipeline` a `cross_val_score` fa sì che il vettorizzatore si adatti **dentro** ogni partizione. Prima, perché stratificare e mescolare: sull'insieme sbilanciato della sezione 6, che è ordinato per categoria.

In [ ]:
des_spa = submuestrear(tickets["spa"], CUOTAS)            # ordinato per categoria
for nombre, particionador in (("KFold senza mescolare", KFold(n_splits=5)),
                              ("StratifiedKFold", StratifiedKFold(n_splits=5, shuffle=True, random_state=42))):
    print(nombre)
    for k, (_, idx_prueba) in enumerate(particionador.split(des_spa, des_spa["categoria"]), 1):
        print("  partizione", k, dict(des_spa.iloc[idx_prueba]["categoria"].value_counts().sort_index()))
skf_test = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for _, idx_prueba in skf_test.split(des_spa, des_spa["categoria"]):
    assert des_spa.iloc[idx_prueba]["categoria"].value_counts()["devoluciones"] == 2      # 10 devoluciones ripartite: 2 per partizione

**Cosa osservi?** Senza mescolare, le partizioni di test hanno composizioni molto diverse: la 2 contiene **solo `envio`** e la 4 e la 5 **solo `facturacion`**, e la 1 non ha nessun `facturacion`; il modello è valutato con classi che ha visto a malapena in addestramento. Con `StratifiedKFold` ogni partizione conserva le proporzioni (circa 20 `facturacion`, 10 `envio`, 4 `cuenta`, 2 `devoluciones`). Ora confrontiamo i tre modelli con 5 partizioni stratificate, sui ticket bilanciati, e l'effetto di `class_weight` su quelli sbilanciati.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
filas = []
for nombre, fabrica in MODELOS.items():
    fila = {"modello": nombre}
    for idioma in ("spa", "ita"):
        puntos = cross_val_score(crear_pipeline(fabrica()), tickets[idioma]["texto"], tickets[idioma]["categoria"], cv=skf)
        fila[f"{idioma} media"] = round(puntos.mean(), 3)
        fila[f"{idioma} dev. standard"] = round(puntos.std(), 3)
        assert len(puntos) == 5 and puntos.mean() > 0.7
    filas.append(fila)
print(pd.DataFrame(filas).to_string(index=False))

print()
print("Insieme sbilanciato (spagnolo), F1 macro con CV stratificata a 5 partizioni:")
for peso in (None, "balanced"):
    puntos = cross_val_score(crear_pipeline(LogisticRegression(max_iter=1000, class_weight=peso)), des_spa["texto"], des_spa["categoria"], cv=skf, scoring="f1_macro")
    print(f"  class_weight={peso}: media {puntos.mean():.3f} ± {puntos.std():.3f}")

**Cosa osservi?** Con la validazione incrociata, i tre modelli stanno tra 0,865 e 0,905 e **la deviazione standard tra partizioni (0,021-0,044) è dello stesso ordine delle differenze tra modelli**: non c'è un vincitore chiaro. È la cifra da riportare (media ± deviazione) e si avvicina alla linea di base ≈ 0,88-0,89 del corso. Nell'insieme sbilanciato, `class_weight="balanced"` migliora l'F1 macro (0,333 → 0,673), il che conferma con più dati di test ciò che suggeriva la partizione singola della sezione 6.

## 9. Riepilogo e passo successivo

- **Naive Bayes multinomiale** = priori × prodotto di `P(parola | classe)`; il **lisciamento di Laplace** (`alpha = 1`) somma uno pseudoconteggio per evitare probabilità zero. È stato calcolato a mano e coincide con `MultinomialNB`.
- **Regressione logistica** e **`LinearSVC`** su TF-IDF rendono in modo simile su questi corpora; con una sola partizione non si distingue il migliore.
- La **`Pipeline`** adatta tutto solo con l'addestramento. Con il TF-IDF la fuga è stata quasi nulla, ma con un passo che usa le etichette (selezione delle caratteristiche) è stata enorme: tutto ciò che impara va dentro.
- **Precisione, richiamo, F1** (macro e ponderato) e **matrice di confusione** dicono dove sbaglia il modello. Con classi sbilanciate l'**accuratezza inganna**: guarda l'F1 macro e il richiamo per classe.
- I **coefficienti** si leggono con senso critico: gli artefatti (verdetti, etichette di oggetto) finiscono tra le parole più pesanti.
- **`StratifiedKFold`** dà una stima più stabile e rispetta le proporzioni delle classi.
- Ricorda: corpora piccoli, sintetici e con spagnolo/italiano quasi paralleli: le cifre sono un *esempio illustrativo*.

**Prossimo passo:** nella lezione 6, *Analisi del sentimento classica*, applicherai questi classificatori a recensioni con negazione e ironia e li confronterai con un lessico. Prima, fai il notebook di pratica di questa lezione.